In [1]:
import boto3
print(f"Boto3 version: {boto3.__version__}")


Boto3 version: 1.43.89


In [2]:
usage_history = []

In [ ]:
import boto3
import logging
import os
# logging.basicConfig(level=logging.DEBUG)
# boto3.set_stream_logger('', logging.DEBUG)

# Create logs directory if it doesn't exist
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

# Configure logging
logging.basicConfig(
    filename=os.path.join(log_dir, "log.txt"),
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Log messages
logging.debug("This is a debug message")
logging.info("This is an info message")
logging.warning("This is a warning message")
logging.error("This is an error message")
logging.critical("This is a critical message")


region = "ap-south-1"
target_model_id = "anthropic.claude-haiku-4-5-20251001-v1:0"
# initialize a boto3 session with the specified region
session = boto3.Session(region_name=region)

bedrock = session.client("bedrock")
# list all inference profiles and filter for the one that matches the target model ID
profiles = bedrock.list_inference_profiles(
    typeEquals="SYSTEM_DEFINED"
)["inferenceProfileSummaries"]

matching_profiles = [
    profile
    for profile in profiles
    if any(
        target_model_id in model.get("modelArn", "")
        for model in profile.get("models", [])
    )
]

if not matching_profiles:
    raise RuntimeError(
        f"No inference profile for {target_model_id} is available in {region}."
    )

model_id = matching_profiles[0]["inferenceProfileId"]
print(f"Using inference profile: {model_id}")



Using inference profile: global.anthropic.claude-haiku-4-5-20251001-v1:0


**claude do not store any messages. SO, for multi turn chat type conversation , the messages have to be maintained/storted locally. Provide this list with every follow up request
**

In [4]:
client = session.client("bedrock-runtime")

In [5]:
def add_user_message(messages, text):

    user_message = {
        "role": "user",
        "content": text,
    }
    messages.append(user_message)
    return user_message

def add_assistant_message(messages, text):

    assistant_message = {
        "role": "assistant",
        "content": text,
    }
    messages.append(assistant_message)
    return assistant_message

def chat_with_model(messages):
    response = client.converse(
        modelId=model_id,
        messages=messages,
    )
    return response["output"]["message"]["content"][0]["text"]


In [6]:
def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "content": [{"text": text}],
    })


def add_assistant_message(messages, text):
    messages.append({
        "role": "assistant",
        "content": [{"text": text}],
    })


def chat_with_model(messages):
    # print(f"Messages being sent to model: {messages}")
    response = client.converse(
        modelId=model_id,
        messages=messages,
    )
    return response["output"]["message"]["content"][0]["text"]


# Make a starting list of messages.
messages = []

# Add the initial user question.
add_user_message(messages, "What's 1+1?")
answer = chat_with_model(messages)
print(answer)

# Add the assistant answer before sending the follow-up.
add_assistant_message(messages, answer)
add_user_message(messages, "And 3 more added to that?")

answer = chat_with_model(messages)
print(f"Assistant : {answer}" )

1 + 1 = 2
Assistant : 2 + 3 = 5


## Chat bot excercise

In [7]:
# input_text = |

In [12]:
messages=[]
user_text = input("You: ").strip()
print(f"User input: {user_text}")



User input: what is aws bedrock in 2 lines


In [13]:
while user_text and user_text != "exit":
    add_user_message(messages, user_text)
    answer = chat_with_model(messages)
    print(f"Assistant: {answer}")
    add_assistant_message(messages, answer)
    user_text = input("You: ").strip()
    print(f"User input: {user_text}")
else:
    print("user endede the conversation.")

Assistant: # AWS Bedrock

AWS Bedrock is a fully managed service that provides access to foundation models (large language models) from various providers like Claude, Llama, and Mistral through a single API. It allows you to build AI applications without managing infrastructure, handling model deployment, or fine-tuning.
User input: te;l more more about the gen ai think in 3 lines
Assistant: # Generative AI with AWS Bedrock

Generative AI uses foundation models trained on vast amounts of data to generate new content like text, images, code, and more based on user prompts. AWS Bedrock simplifies accessing these powerful models by eliminating infrastructure management, allowing developers to focus on building innovative AI applications quickly. It supports various use cases including chatbots, content creation, code generation, and document analysis through a unified API.
User input: ok, what is sagemaker
Assistant: # AWS SageMaker

AWS SageMaker is a fully managed machine learning servi

In [10]:
import os
print(os.getcwd())

f:\GenAI\Spec-driven-development\anthropic-bedrock\spec_driven_development\claude_bedrock_lesson
